In [0]:
# Imports and Variable Set Up
import os
import logging
import uuid
import time
import requests
import json
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from databricks.vector_search.client import VectorSearchClient
from openai import OpenAI


logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Catalog / Schema / Volume
catalog = "workspace"
schema = "ai_project"
volume = "raw_data"

# Base Volume Path
vol_path = f"/Volumes/{catalog}/{schema}/{volume}/"

# TODO: Update path usage below with appropriate volume paths
# landing_path = vol_path + "raw"
# processed_path = vol_path + "processed"

# Books paths
books_landing_path = vol_path + "books/raw"
books_processed_path = vol_path + "books/processed"

# Docs paths
docs_landing_path = vol_path + "docs/raw"
docs_processed_path = vol_path + "docs/processed"

# AI Gateway / Model config
base_url = "https://7474648118426063.ai-gateway.cloud.databricks.com/mlflow/v1"
embedding_model = "databricks-bge-large-en"
llm_model = "databricks-meta-llama-3-1-405b-instruct"       # TODO: Remove/replace var declarations below

# Vector Search config
endpoint_name = "book_search_endpoint"
index_name = f"{catalog}.{schema}.book_vector_index"
# silver_table = f"{catalog}.{schema}.processed_chunks"
books_chunks_table = f"{catalog}.{schema}.books_chunks"
docs_chunks_table = f"{catalog}.{schema}.docs_chunks"

# File config
valid_extensions = ('.txt', '.pdf')
source_config = {
    "books": {
        "landing_path": books_landing_path,
        "processed_path": books_processed_path,
        "table": books_chunks_table,
        "min_size_kb": 10
    },
    "docs": {
        "landing_path": docs_landing_path,
        "processed_path": docs_processed_path,
        "table": docs_chunks_table,
        "min_size_kb": 1
    }
}

# %pip install -r https://raw.githubusercontent.com/seaninc-training/databricks-ai-project/refs/heads/main/requirements.txt
# dbutils.library.restartPython()

In [0]:
all_files_to_process = {}

def get_files_to_process(landing_path, source_type):

    if source_type is None:
        raise ValueError("source_type must be specified: 'books' or 'docs'")
    elif source_type not in source_config:
        raise ValueError(f"Invalid source_type '{source_type}'. Must be one of: {list(source_config.keys())}")
    
    to_process = []
    min_size_kb = source_config[source_type]["min_size_kb"]
    
    try:
        raw_files = dbutils.fs.ls(landing_path)
        if not raw_files:
            logger.warning(f"⚠️ Source folder is empty: {landing_path}")
        else:
            logger.info(f"✅ Found {len(raw_files)} files to process.")
            for file in raw_files:
                if file.name.lower().endswith(valid_extensions):
                    size_kb = file.size / 1024
                    if size_kb < min_size_kb:
                        logger.error(f"❌ Skipping {file.name}: File is too small ({size_kb:.2f} KB).")
                        continue
                    to_process.append({
                        "path": file.path,
                        "name": file.name,
                        "type": "pdf" if file.name.lower().endswith(".pdf") else "text",
                        "source_type": source_type
                    })
                    logger.info(f"📖 {file.name} validated ({size_kb:.2f} KB).")
                else:
                    logger.warning(f"⚠️  {file} is not a permitted file type.")

            if len(to_process) == 0:
                logger.warning(f"⚠️ No valid files found in {landing_path}.")
            else:
                logger.info(f"✅ Total files ready for ingestion: {len(to_process)}")

    except Exception as e:
        logger.error(f"❌ Error accessing volume: {e}")
    
    return to_process


for source_type, config in source_config.items():
    all_files_to_process[source_type] = get_files_to_process(config["landing_path"], source_type)

In [0]:
# Set up text splitter for chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    add_start_index=True,
    separators=["\n\n", "\n", ". ", " ", ""])


def process_files(files_to_process, source_type):
    config = source_config[source_type]
    chunks_table = config["table"]
    processed_path = config["processed_path"]

    for file_info in files_to_process:
        try:
            logger.info(f"🚀 Processing: {file_info['name']}")

            local_path = file_info['path'].replace("dbfs:", "")
            
            if file_info['type'] == "pdf":
                loader = PyPDFLoader(local_path)
            else:
                loader = TextLoader(local_path, encoding="utf-8")

            raw_docs = loader.load()

            # Create Chunks
            chunks = text_splitter.split_documents(raw_docs)
            
            # Prepare data for Vector Search
            data = [{
                "chunk_id": str(uuid.uuid4()),
                "content": chunk.page_content, 
                "source": file_info['name'],
                "source_type": file_info['source_type'],
                "type": file_info['type'],
                "page_number": chunk.metadata.get("page", 1),
                "start_index": chunk.metadata.get("start_index", 0)
            } for chunk in chunks]

            # Convert to Spark DF 
            df = spark.createDataFrame(data)

            # Write to Delta with CDF Enabled
            if not spark.catalog.tableExists(chunks_table):
                (df.write.format("delta")
                   .option("delta.enableChangeDataFeed", "true")
                   .mode("overwrite")
                   .saveAsTable(chunks_table))
                logger.info(f"✨ Created new table: {chunks_table}")
            else:
                df.write.format("delta").mode("append").saveAsTable(chunks_table)
                logger.info(f"➕ Appended {len(chunks)} chunks to {chunks_table}")

            # Move file to processed folder
            destination = f"{processed_path}/{file_info['name']}"
            dbutils.fs.mv(local_path, destination)
            logger.info(f"✅ Processed and moved: {file_info['name']}")

        except Exception as e:
            logger.error(f"❌ Failed to process {file_info['name']}: {e}")

    logger.info(f"🏁 All files processed for table: {chunks_table}")


for source_type, files in all_files_to_process.items():
    if files:
        process_files(files, source_type)
    else:
        logger.info(f"ℹ️ No files to process for source type: {source_type}")

In [0]:
# Cell 4: Setup Vector Search Endpoint
endpoint_name = "book_search_endpoint"
vsc = VectorSearchClient()

if vsc.endpoint_exists(endpoint_name):
    logger.info(f"✅ Endpoint '{endpoint_name}' already exists.")
else:
    # Create the endpoint if it doesn't exist
    # This is a 'Standard' endpoint (best for general RAG projects)
    try:
        logger.info(f"🚀 Creating endpoint '{endpoint_name}'.")
        vsc.create_endpoint(name=endpoint_name, endpoint_type="STANDARD")
        vsc.wait_for_endpoint(endpoint_name)
    except Exception as e:
        logger.error(f"❌ Failed to create endpoint: {e}")

# Wait for it to be ready before moving to the next cell
logger.info(f"🟢 Endpoint '{endpoint_name}' is now ONLINE.")

In [0]:
# Unified Cell 5: Idempotent Vector Setup

# vsc = VectorSearchClient()
index_name = f"{catalog}.{schema}.book_vector_index"
embedding_model = "databricks-bge-large-en"

# 1. Ensure Endpoint exists
if not any(e['name'] == endpoint_name for e in vsc.list_endpoints().get('endpoints', [])):
    logger.info(f"🚀 Creating endpoint '{endpoint_name}'...")
    try:
        vsc.create_endpoint(name=endpoint_name, endpoint_type="STANDARD")
        vsc.wait_for_endpoint(endpoint_name)
        logger.info(f"🟢 Endpoint '{endpoint_name}' is now ONLINE.")
    except Exception as e:
        logger.error(f"❌ Failed to create endpoint: {e}")
else:
    logger.info(f"✅ Endpoint '{endpoint_name}' is ready.")

# 2. Ensure Index exists
if not vsc.index_exists(endpoint_name=endpoint_name, index_name=index_name):
    logger.info(f"✨ Creating new index '{index_name}'...")
    try:
        vsc.create_delta_sync_index(
            endpoint_name=endpoint_name,
            source_table_name=silver_table, 
            index_name=index_name,
            pipeline_type="TRIGGERED",
            primary_key="chunk_id",        
            embedding_source_column="content",
            embedding_model_endpoint_name=embedding_model
        )
        # Creation automatically triggers the first sync
        logger.info("⏳ Initial sync started automatically.")
    except Exception as e:
        logger.error(f"❌ Failed to create index: {e}")
else:
    logger.info(f"✅ Index '{index_name}' already exists.")
    
    # 3. Smart Sync: Only sync if we actually processed new files today
    if len(to_process) > 0:
        try:
            logger.info(f"🔄 New data detected ({len(to_process)} files). Triggering sync...")
            vsc.get_index(endpoint_name, index_name).sync()
        except Exception as e:
            logger.error(f"❌ Failed to sync index: {e}")
    else:
        logger.info("ℹ️ No new files in 'raw' folder. Skipping sync to save time.")

# 4. Wait for Index to be queryable (Online)
logger.info("📡 Checking index status...")
while True:
    status = vsc.get_index(endpoint_name, index_name).describe().get("status", {})
    state = status.get("detailed_state", "UNKNOWN")
    if "ONLINE" in state:
        logger.info(f"🟢 Index is ONLINE. Ready for search!")
        break
    elif "FAILED" in state:
        logger.error(f"❌ Index reached a failure state: {state}. Check the Sync History in Catalog Explorer.")
        break
    logger.info(f"⏳ Index state: {state}... (Waiting 30s)")
    time.sleep(30)

In [0]:
# Cell 6: Semantic Search (The Manual Diagnostic)
query = "What was the weapon Raskolnikov used in the crime?" 

try:
    # 1. Grab the index object
    index = vsc.get_index(endpoint_name, index_name)

    # 2. Manual Search
    results = index.similarity_search(
        query_text=query,
        columns=["content", "source", "page_number", "start_index"],
        num_results=4
    )

    docs = results.get('result', {}).get('data_array', [])

    print(f"\n📡 RAW VECTOR SEARCH TEST: '{query}'")
    print("="*70)

    if not docs:
        print("⚠️ No matches found in the Vector Index.")
    else:
        for i, doc in enumerate(docs):
            # Mapping indices to your updated columns list
            content, source, page, start_idx = doc[0], doc[1], doc[2], doc[3]

            print(f"📍 [Match {i+1}] | File: {source} | Page: {page} | Index: {start_idx}")
            print(f"📄 \"{content[:300]}...\"")
            print("-" * 50)

except Exception as e:
    logger.error(f"❌ Diagnostic failed: {e}")

In [0]:
# Initialize Chat History
if 'chat_history' not in globals():
    chat_history = []
    logger.info("🧠 Memory Initialized.")

# 1. Setup the Client
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url=base_url
)

def search_documents(question):
    """
    A generic RAG search function that queries the Vector Index 
    and returns raw context with citations.
    """
    # --- 1. LIBRARIAN: RETRIEVAL ---
    search_results = index.similarity_search(
        query_text=question, 
        columns=["content", "source", "page_number", "start_index"], 
        num_results=4 
    )
    res_data = search_results.get('result', {}).get('data_array', [])
    
    # Build context with all metadata
    context_blocks = [
        f"Source Page {row[2]} (File: {row[1]}, Index: {row[3]}): {row[0]}" 
        for row in res_data
    ]
    context = "\n---\n".join(context_blocks)
    
    # DEBUG: See what the Librarian found
    logger.info(f"📡 LIBRARIAN FOUND {len(res_data)} excerpts for query: '{question}'")
    
    return context

# Define the Tool Schema for the Agent
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_documents",
            "description": "Use this tool to retrieve specific facts or technical context from the uploaded documentation/books.",
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {"type": "string", "description": "The search query."}
                },
                "required": ["question"]
            }
        }
    }
]




In [0]:
def run_agent(user_query):
    global chat_history
    
    # --- 1. SET THE PERSONALITY & CONSTRAINTS ---
    # We moved your Operating Rules here!
    system_prompt = {
        "role": "system", 
        "content": """You are a Technical Research Assistant. 
        RULES:
        1. Only use provided tool data. If the tool finds nothing, state that clearly.
        2. Always cite 'Source Page' and 'File Name'.
        3. If a definitive motive or answer isn't in the data, explain what IS there but clarify the uncertainty."""
    }

    # Build the full conversation thread
    messages = [system_prompt] + chat_history + [{"role": "user", "content": user_query}]

    # --- 2. INITIAL CALL (Thinking Phase) ---
    response = client.chat.completions.create(
        model="databricks-meta-llama-3-1-405b-instruct",
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

    # --- COST TRACKER (Part 1) ---
    usage = response.usage
    total_tokens = usage.total_tokens
    
    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls

    # --- 3. TOOL EXECUTION (Acting Phase) ---
    if tool_calls:
        for tool_call in tool_calls:
            query_args = json.loads(tool_call.function.arguments)
            observations = search_documents(query_args['question'])
            
            # Feed observations back to the "Brain"
            messages.append(response_message)
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": "search_documents",
                "content": observations
            })
            
            # Final Generation
            final_response = client.chat.completions.create(
                model="databricks-meta-llama-3-1-405b-instruct",
                messages=messages
            )
            
            # --- COST TRACKER (Part 2) ---
            total_tokens += final_response.usage.total_tokens
            answer = final_response.choices[0].message.content
    else:
        answer = response_message.content

    # --- 4. UPDATE MEMORY ---
    chat_history.append({"role": "user", "content": user_query})
    chat_history.append({"role": "assistant", "content": answer})

    # Display Cost and Answer
    print(f"\n{'='*50}")
    print(f"🤖 AGENT RESPONSE:\n{answer}")
    print(f"{'='*50}")
    print(f"💰 USAGE: {total_tokens} tokens")

    return answer

# Test
print(run_agent("What weapon did Raskolnikov use?"))


In [0]:
print(run_agent("Can you tell me how many documents you have access to?"))

In [0]:
print(run_agent("Since you have only 1 document, can you describe the contents of that document? What is it?"))

In [0]:
run_agent("Do you know why Raskolnikov committed his murders?")

In [0]:
run_agent("Based on the document alone, what was the author trying to communicate through this book?")

In [0]:
# def ask_scholar(question):
#     global chat_history
    
#     # --- 1. LIBRARIAN: RETRIEVAL ---
#     search_results = index.similarity_search(
#         query_text=question, 
#         columns=["content", "source", "page_number", "start_index"], 
#         num_results=4 
#     )
#     res_data = search_results.get('result', {}).get('data_array', [])
    
#     # Build context with your metadata fix
#     # row[0] = content
#     # row[1] = source
#     # row[2] = page_number
#     # row[3] = start_index
#     context_blocks = [f"Source Page {row[2]} (File: {row[1]}): {row[0]}" for row in res_data]
#     context = "\n---\n".join(context_blocks)

#     # --- 2. ORCHESTRATION: PREPARE MESSAGES ---
#     # Start with the System Prompt (The Rules)
#     messages = [
#         {
#             "role": "system", 
#             "content": """You are a Technical Research Assistant. Your goal is to provide accurate answers based solely on the provided context excerpts. Use the 'Why' logic to explain motivations or causes found in the text, whether they are philosophical, technical, or financial.
            
#             OPERATING RULES:
#             1. SCANNING: Search the text for any object used to strike, cut, or kill. Prioritize mentions where an object is associated with blood.
#             2. THE 'WHY': Look for mentions of the criminal's mental state, financial condition, or philosophical theories.
#             3. EVIDENCE: If you find a weapon, describe the specific actions mentioned in the excerpts.
            
#             STRICT CONSTRAINTS:
#             4. NO OUTSIDE KNOWLEDGE: Use ONLY the provided excerpts. If the weapon isn't named in the text, say 'The provided excerpts do not name the weapon.'
#             5. CITATIONS: Always cite the 'Source Page' and 'File Name' for every piece of information provided.
#             6. UNCERTAINTY: If the 'Why' isn't explicitly in the context, explain what IS there (like his poverty) but clarify that a definitive motive isn't stated."""
#         }
#     ]
    
#     # Add the "Past" (The Memory)
#     # We use .extend() to add the list of previous turns
#     messages.extend(chat_history)
        
#     # Add the "Present" (New Context + Current Question)
#     user_input = f"NEW CONTEXT EXCERPTS:\n{context}\n\nUSER QUESTION: {question}"
#     messages.append({"role": "user", "content": user_input})

#     # --- 3. SCHOLAR: GENERATION ---
#     response = client.chat.completions.create(
#         model="databricks-meta-llama-3-1-405b-instruct",
#         messages=messages,
#         max_tokens=1024,
#         temperature=0.1 # Keep it focused and "scholarly"
#     )
    
#     answer = response.choices[0].message.content
    
#     # --- 4. RECORD: UPDATE HISTORY ---
#     # We only store the core Q&A to save tokens
#     chat_history.append({"role": "user", "content": question})
#     chat_history.append({"role": "assistant", "content": answer})
    
#     return answer

# print(f"📖 SCHOLAR RESPONSE:\n{ask_scholar('What weapon did Raskolnikov use and why?')}")